# RAG - Retrieval-Augmented Generation
É uma técnica que combina o poder de um modelo de linguagem (LLM) com sistemas de recuperação de informações. Em termos simples, o RAG permite que o LLM acesse dados externos, como seus próprios documentos, bases de conhecimento ou a internet, antes de gerar uma resposta.

O RAG **resolve três grandes problemas** dos LLMs tradicionais:

1. **Conhecimento Desatualizado**: LLMs são treinados em grandes volumes de dados, mas esse conhecimento é estático e limitado à data do treinamento. O RAG permite que o modelo acesse informações em tempo real e dados que são constantemente atualizados.

2. **Alucinações**: Como os LLMs às vezes inventam informações para preencher lacunas, eles podem gerar respostas incorretas ou sem fundamento. O RAG "aterra" a resposta em fatos concretos, usando as informações recuperadas de uma fonte externa confiável, o que reduz drasticamente a chance de alucinações.

3. **Falta de Transparência**: Com o RAG, o modelo não apenas responde, mas também pode citar as fontes de onde a informação foi extraída. Isso aumenta a confiança do usuário, pois ele pode verificar a veracidade da resposta.

Como o RAG funciona?


## Libraries

In [1]:
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
import datetime
import time
import requests


# # Modelos LLM (Large Language Models)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.language_models.chat_models import BaseChatModel


from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

from langchain_core.output_parsers import StrOutputParser


# Criação e execução de agentes
from langchain_classic.agents import( 
Tool, 
AgentExecutor,
create_tool_calling_agent,
create_react_agent)

# # Ferramentas customizadas para agentes
from langchain.tools import tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_experimental.tools.python.tool import PythonAstREPLTool

from langchain_classic.memory import ConversationBufferMemory


from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownHeaderTextSplitter

# # Componentes de RAG (Retrieval-Augmented Generation)
from langchain_chroma import Chroma  # Armazenamento vetorial





In [2]:
# 1. Configuração do diretório de saída
OUTPUT_DOCUMENTS_DIR: str = './documentos/'
os.makedirs(OUTPUT_DOCUMENTS_DIR, exist_ok=True)

# 2. Carregamento das variáveis de ambiente (.env)
ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()


llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))

print("✔ LLM carregada:",llm_gemini.profile['name'])


✔ OUTPUT_DOCUMENTS_DIR: ./documentos/
✔ Variáveis de ambiente carregadas do arquivo .env
✔ LLM carregada: Gemini 3.1 Flash Lite Preview


## Criando Banco de dados

In [4]:

# Visualizar os detalhes da execução
def cria_banco_de_dados_vetorial(path_documentos:str,chk_sz=600,chk_ov=100) -> None:
    try:
        # Carrega os documentos do diretório especificado
        documents = PyPDFDirectoryLoader(path_documentos).load()
        
        # Usando embeddings gemini
        embeddings = GoogleGenerativeAIEmbeddings(
            model="gemini-embedding-001",
            google_api_key=os.getenv("GOOGLE_API"))

        # Cria um banco de dados vetorial usando Chroma
        split_documents = RecursiveCharacterTextSplitter(chunk_size=chk_sz, chunk_overlap=chk_ov).split_documents(documents)

        # Cria o banco de dados vetorial
        vectorstore = Chroma.from_documents(split_documents, embeddings, persist_directory=f'{OUTPUT_DOCUMENTS_DIR}vectorstore')

        print("✔ Banco de dados vetorial criado com sucesso.")
    except Exception as e:
        print(f"Erro ao carregar documentos: {e}")

# add documento para criação do banco de dados
OUTPUT_DOCUMENTS_DIR: str = './documentos/'
cria_banco_de_dados_vetorial(path_documentos=OUTPUT_DOCUMENTS_DIR)

✔ Banco de dados vetorial criado com sucesso.


## Carregando o banco vetorial criado

In [5]:
def carrega_banco_de_dados_vetorial(path_documentos:str) -> Chroma:
    try:
        # Usando embeddings gemini
        embeddings = GoogleGenerativeAIEmbeddings(
            model="gemini-embedding-001",
            google_api_key=os.getenv("GOOGLE_API"))
        
        vectorstore = Chroma(persist_directory=path_documentos, embedding_function=embeddings)
        if vectorstore:
            print('banco de dados carregado!')
        
        return vectorstore
    except Exception as e:
        print(f"Erro ao carregar o banco de dados vetorial: {e}")
        return None

In [7]:
vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore')
# docs = None
# retriever = vectorstore.as_retriever()
# docs = retriever.invoke("Data H")


banco de dados carregado!


## criando contexto com a base de dados

In [6]:
def busca_na_base_de_documentos(pergunta:str) -> str:
    """Use esta ferramenta para responder perguntas sobre a Data H, seus produtos como NIC, Consultoria, Cyber Segurança,
       ou qualquer informação contida na base de conhecimento. A entrada deve ser a pergunta do usuário."""
    vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore')
    contexto = None
    if vectorstore:
        retriever = vectorstore.as_retriever()
        docs = retriever.invoke(pergunta)
        contexto = "\n\n".join([doc.page_content for doc in docs])
    return contexto


## Criando agente RAG

In [8]:
def string_gemini(out_agent_exe):
    if isinstance(out_agent_exe, list) and len(out_agent_exe) > 0:
        if isinstance(out_agent_exe[0], dict) and 'text' in out_agent_exe[0]:
            return out_agent_exe[0]['text']
    
    return  out_agent_exe[0]['text']


def agente_langchain_RAG1(modelo_llm=llm_gemini) -> dict:
    ferramentas = []
    memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True, input_key="input")

    prompt = PromptTemplate(
        input_variables=["input", "context", "chat_history", "agent_scratchpad"],
        template=""" {chat_history}
                Você é um agente de IA especializado em responder perguntas.
                Contexto: {context}
                Pergunta: {input}
                {agent_scratchpad}
        """)

    agente = create_tool_calling_agent(modelo_llm,ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas, memory=memoria)
    return executor_do_agente

#### Executando sem contexto

In [10]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "O que é o NIC?"
contexto1 = ''
resposta1 = executor_do_agente.invoke({"input": pergunta1, "context": contexto1})
resposta1=string_gemini(resposta1['output'])

print(resposta1)
print('\n' + '=' * 10)

O termo **NIC** pode se referir a diferentes conceitos, dependendo da área de atuação. Os dois significados mais comuns são:

### 1. Na Tecnologia (Redes de Computadores)
**NIC** significa *Network Interface Card* (em português, **Placa de Interface de Rede**).
*   **O que é:** É um componente de hardware (uma placa ou chip) que permite que um computador ou dispositivo se conecte a uma rede (como a internet ou uma rede local).
*   **Função:** Ela atua como a interface física entre o computador e o meio de transmissão (cabo Ethernet, Wi-Fi, fibra óptica). Cada NIC possui um endereço único chamado **endereço MAC**, que identifica o dispositivo na rede.

### 2. Na Administração Pública (Brasil)
**NIC.br** significa *Núcleo de Informação e Coordenação do Ponto BR*.
*   **O que é:** É o braço executivo do **CGI.br** (Comitê Gestor da Internet no Brasil).
*   **Função:** É a entidade responsável por implementar as decisões e projetos do CGI.br, incluindo o registro de domínios ".br" (através

In [11]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "O que é o NIC?"
contexto2 = busca_na_base_de_documentos(pergunta1)
resposta2 = executor_do_agente.invoke({"input": pergunta1, "context": contexto2})
resposta2=string_gemini(resposta2['output'])
#
print(resposta2)
print('\n' + '=' * 10)

banco de dados carregado!
O **NIC (Núcleo de Inteligência Contínua)** é uma ferramenta estratégica desenvolvida pela DATA H, essencial para empresas modernas. Ele atua rastreando o mercado para antecipar tendências e inovações, permitindo que as organizações se mantenham competitivas, ágeis e em constante crescimento.

Em suma, o NIC funciona como um parceiro estratégico que não apenas automatiza decisões, mas arquiteta inteligências capazes de aprender com o tempo, adaptar-se a imprevistos e proteger o que é prioritário para o negócio.

